In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
# Gold Layer — Star Schema (matching Databricks logic exactly)
from pyspark.sql.functions import col, avg, coalesce, lit, dense_rank, when, trim
from pyspark.sql.window import Window


# 1. Load Silver Tables
jobs        = spark.read.table("silver.jobs")
locations   = spark.read.table("silver.locations")
occupations = spark.read.table("silver.occupations")
channels    = spark.read.table("silver.channels")
messages    = spark.read.table("silver.messages")
norm_skills = spark.read.table("silver.norm_skills")

# 2. Build Dimensions & Bridge Tables
dim_skills = norm_skills.select(col("skill_id"), col("skill").alias("skill_name")).distinct()
bridge_job_skills = norm_skills.select(col("job_id"), col("skill_id")).distinct()

# FIX #7: Clean is_remote_work — validate only "0"/"1" values
locations_clean = locations.withColumn(
    "is_remote_work",
    when(~trim(col("is_remote_work").cast("string")).isin("0", "1"), None)
    .otherwise(col("is_remote_work"))
)

dim_locations = (
    locations_clean
    .select("country", "city", "is_remote_work")
    .filter(
        col("city").isNotNull() &
        col("country").isNotNull() &
        col("is_remote_work").isNotNull()
    )
    .distinct()
    .withColumn("location_id", dense_rank().over(Window.orderBy("country", "city", "is_remote_work")))
    .select("location_id", "country", "city", "is_remote_work")
)

dim_occupations = (
    occupations
    .select("occupation")
    .filter(col("occupation").isNotNull())
    .distinct()
    .withColumn("occupation_id", dense_rank().over(Window.orderBy("occupation")))
    .select("occupation_id", col("occupation").alias("occupation_name"))
)

dim_channels = (
    channels
    .select("title", "username", "member_count")
    .filter(col("title").isNotNull())
    .distinct()
    .withColumn("channel_id", dense_rank().over(Window.orderBy("title")))
    .select("channel_id", col("title").alias("channel_name"), "username", "member_count")
)

# 3. Salary Imputation
# FIX #8: Join ALL occupations per job (no row_number dedup) — matches Databricks
occ_for_join = occupations.select(
    col("job_id").alias("occ_job_id"),
    "occupation"
).filter(col("occupation").isNotNull())

jobs_with_occ = jobs.join(
    occ_for_join,
    jobs.id == col("occ_job_id"),
    "left"
).drop("occ_job_id")

avg_by_occ = (
    jobs_with_occ
    .filter(col("cleaned_salary").isNotNull())
    .groupBy("occupation")
    .agg(avg("cleaned_salary").alias("avg_salary_by_occ"))
)

overall_avg = jobs.filter(col("cleaned_salary").isNotNull()).agg(avg("cleaned_salary")).collect()[0][0]

jobs_imputed = (
    jobs_with_occ
    .join(avg_by_occ, on="occupation", how="left")
    .withColumn("cleaned_salary",
        coalesce(
            col("cleaned_salary"),
            col("avg_salary_by_occ"),
            lit(overall_avg)
        )
    )
    .drop("avg_salary_by_occ", "occupation")
)

# 4. Build Fact Table
# FIX #9: Use locations_clean (with cleaned is_remote_work) for the join
jobs_s1 = jobs_imputed.join(
    locations_clean.select(col("job_id").alias("loc_job_id"), "country", "city", "is_remote_work"),
    jobs_imputed.id == col("loc_job_id"),
    "left"
).drop("loc_job_id")

jobs_s2 = jobs_s1.join(
    dim_locations,
    on=["country", "city", "is_remote_work"],
    how="left"
).drop("country", "city", "is_remote_work")

# FIX #10: Use ALL occupations (no dedup) for occupation_id — matches Databricks
occ_lookup = occupations.select(
    col("job_id").alias("occ_job_id"),
    col("occupation")
)

jobs_s3 = jobs_s2.join(
    occ_lookup,
    jobs_s2.id == col("occ_job_id"),
    "left"
).drop("occ_job_id")

jobs_s4 = jobs_s3.join(
    dim_occupations,
    jobs_s3.occupation == dim_occupations.occupation_name,
    "left"
).drop("occupation", "occupation_name")

# Channel ID via messages → channels → dim_channels
msg_lookup = messages.select(col("id").alias("msg_lookup_id"), col("chat_id").alias("msg_chat_id"))
jobs_s5 = jobs_s4.join(msg_lookup, jobs_s4.message_id == col("msg_lookup_id"), "left").drop("msg_lookup_id")

chan_lookup = channels.select(col("chat_id").alias("chan_chat_id"), col("title").alias("chan_title"))
jobs_s6 = jobs_s5.join(chan_lookup, jobs_s5.msg_chat_id == col("chan_chat_id"), "left").drop("msg_chat_id", "chan_chat_id")

jobs_s7 = jobs_s6.join(
    dim_channels.select("channel_id", "channel_name"),
    jobs_s6.chan_title == dim_channels.channel_name,
    "left"
).drop("chan_title", "channel_name")

fact_jobs = jobs_s7.select(
    col("id").alias("job_id"),
    "job_name", "company_name", "job_type",
    col("cleaned_salary").alias("salary"),
    "job_date", "input_language", "is_active",
    "location_id", "occupation_id", "channel_id"
).distinct()

# 5. Save to Gold Schema
dim_locations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_locations")
dim_occupations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_occupations")
dim_skills.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_skills")
dim_channels.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_channels")
bridge_job_skills.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.bridge_job_skills")
fact_jobs.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_jobs")

print(f"fact_jobs: {fact_jobs.count()} rows")
print(f"  salary NULL:        {fact_jobs.filter(col('salary').isNull()).count()}")
print(f"  location_id NULL:   {fact_jobs.filter(col('location_id').isNull()).count()}")
print(f"  occupation_id NULL: {fact_jobs.filter(col('occupation_id').isNull()).count()}")
print(f"  channel_id NULL:    {fact_jobs.filter(col('channel_id').isNull()).count()}")
print("--- Gold Layer Completed ---")

StatementMeta(, 5e5c1455-2ebc-41ab-9126-7d935bed9a14, 3, Finished, Available, Finished, False)

--- Gold Layer Completed ---
